In [1]:
import os, re
import numpy as np
import pandas as pd
import ipywidgets as widgets
from ipywidgets import Layout, HBox
from IPython.display import display

pd.set_option('display.max_rows', 100)

def grepNumber(string):
    return [float(s) for s in re.findall(r'-?\d+\.?\d*', string)]

def processData(file):
    global df_OD, df_GFP, df_well, df_Stats
    
    df = pd.read_excel(file)
    idx1 = np.where(df.iloc[:, 0] == 'absorbance')[0][0]
    idx2 = np.where(df.iloc[:, 0] == 'GFP')[0][0]
    
    df_OD = df.iloc[idx1+1:idx2-1, :]
    df_OD.columns = df_OD.iloc[0, :].values
    df_OD = df_OD[1:]
    df_OD = df_OD.set_index(df_OD.iloc[:, 0])
    df_OD = df_OD.iloc[:, 1:]
    df_OD = df_OD.astype('float')
    df_OD = df_OD.dropna(axis=1)
    
    df_GFP = df.iloc[idx2+1:-2, :]
    df_GFP = df_GFP[~df_GFP.iloc[:, 1].isnull()]
    df_GFP.columns = df_GFP.iloc[0, :].values
    df_GFP = df_GFP[1:]
    df_GFP = df_GFP.set_index(df_GFP.iloc[:, 0])
    df_GFP = df_GFP.iloc[:, 1:]
    df_GFP = df_GFP.astype('float')
    df_GFP = df_GFP.dropna(axis=1)
    
    df_well = pd.read_excel(file, sheet_name=1, index_col=0)
    for i in df_well.index:
        for c in df_well.columns:
            if f'{i}{c}' not in df_OD.columns:
                df_well.loc[i, c] = ''
    
    df_Stats = pd.DataFrame([StatsWell(key) for key in df_OD.columns[2:]], index=df_OD.columns[2:])
    df_Stats.columns = ['GrowthRate', 'DoublingTime', 'r_value', 'GFPintensity']
    df_Stats['Strain'] = [df_well.loc[well[0], int(well[1:])] for well in df_OD.columns[2:]]
    df_Stats['GFPintensity'] = np.log10(df_Stats['GFPintensity'])
    
    return None

In [2]:
from scipy import stats
from scipy import interpolate

def blank(series):
    return series - series.min()

def get_TrueLogPhase(indices):
    detect = 0
    for n in range(len(indices)-2):
        values = indices.iloc[n:n+2]
        # print('\n', n, '\n', values)
        if ~values.values.any() and detect==1:
            indices.iloc[n+2:] = False
            break
            # print(n)
        elif values.values.any():
            detect = 1

    return indices

def StatsWell(well, show=False):
    try:
        ts = df_OD['Time [s]']
        ts = ts/3600
        ODs = blank(df_OD[well])
        GFPs = blank(df_GFP[well])
        indices = (ODs>0.11) & (ODs<0.44)
        # print(indices)
        indices = get_TrueLogPhase(indices)
        # print(indices)

        slope, intercept, r_value, p_value, std_err = stats.linregress(ts[indices], np.log(ODs[indices]))
        GrowthRate = slope
        DoublingTime = np.log(2)/slope*60

        GFP_interp = interpolate.interp1d(ODs[indices], GFPs[indices])
        GFPintensity = int(GFP_interp(0.22))

        if show:
            print(f'Growth rate: {round(GrowthRate, 2)} (hr^-1)')
            print(f'Doubling time: {round(DoublingTime, 1)} (min)')
            print(f'r = {round(r_value, 4)}')
            print(f'GFP/OD (at OD=0.22): {GFPintensity/0.22}')

        return GrowthRate, DoublingTime, r_value, GFPintensity
    
    except:
        return np.nan, np.nan, np.nan, np.nan

In [3]:
import matplotlib.pyplot as plt

def Plot_OD_vs_time(wells, ymax=None, file=''):
    ts = df_OD['Time [s]']/3600
    plt.figure(figsize=(8, 4), dpi=600)
    plt.title('OD vs time')
    if ymax: plt.ylim(0, ymax)
    
    for w in wells:
        plt.plot(ts, blank(df_OD[w.split(':')[0]]), label=w)
    
    plt.legend(bbox_to_anchor=(1.3, 1))
    plt.tight_layout()
    if file: plt.savefig(f'{file}_1.png')
    plt.show()
    return None

def Plot_GFP_vs_time(wells, ymax=None, file=''):
    ts = df_OD['Time [s]']/3600
    plt.figure(figsize=(8, 4), dpi=600)
    plt.title('GFP vs time')
    if ymax: plt.ylim(0, ymax)
    
    for w in wells:
        plt.plot(ts, blank(df_GFP[w.split(':')[0]]), label=w)
    
    plt.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
    plt.legend(bbox_to_anchor=(1.3, 1))
    plt.tight_layout()
    if file: plt.savefig(f'{file}_2.png')
    plt.show()
    return None

def Plot_GFP_vs_OD(wells, ymax=None, file=''):
    ts = df_OD['Time [s]']/3600
    plt.figure(figsize=(8, 4), dpi=600)
    plt.title('GFP intensity vs OD')
    if ymax: plt.ylim(0, ymax)
    
    for w in wells:
        plt.plot(blank(df_OD[w.split(':')[0]]), blank(df_GFP[w.split(':')[0]])/blank(df_OD[w.split(':')[0]]), label=w)
    
    plt.axvline(0.22, linestyle='dotted', color='black', alpha=.5)
    plt.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
    plt.legend(bbox_to_anchor=(1.3, 1))
    plt.tight_layout()
    if file: plt.savefig(f'{file}_3.png')
    plt.show()
    return None

In [4]:
files = [f for f in os.listdir() if ('.xlsx' in f)]
widget_file = widgets.Dropdown(options=files, value=files[-1], description='Data Source: ', layout=Layout(height='30px', width='50%'))
load_button = widgets.Button(description="Load", layout=Layout(height='30px', width='15%'))
output_button = widgets.Button(description="Output Processed Table", layout=Layout(height='30px', width='35%'))
output = widgets.Output()

display(HBox([widget_file, load_button, output_button]), output)

def load_on_button_clicked(b):
    try:
        processData(widget_file.value)
        with output:
            print("Load Sucessfully!")
    except:
        with output:
            print("Load Error!")
    return None

def output_on_button_clicked(b):
    try:
        processData(widget_file.value)
        gb = df_Stats.groupby('Strain')
        gb.mean().to_csv(f'{widget_file.value.split(".")[0]}_mean.tsv', sep='\t')
        gb.std().to_csv(f'{widget_file.value.split(".")[0]}_std.tsv', sep='\t')
        with output:
            print("Output Sucessfully!")
    except:
        with output:
            print("Output Error!")
    return None

load_button.on_click(load_on_button_clicked)
output_button.on_click(output_on_button_clicked)

Output()

In [10]:
wells = widgets.SelectMultiple(options=df_Stats.index+': '+df_Stats['Strain'], description='Select Wells:', disabled=False, layout=Layout(height=f'{len(df_Stats)*20}px'))
ymax1 = widgets.IntText(description='No.1 ymax:', disabled=False)
ymax2 = widgets.IntText(description='No.2 ymax:', disabled=False)
ymax3 = widgets.IntText(description='No.3 ymax:', disabled=False)
plot_button = widgets.Button(description='Plot')
figure_name = widgets.Text(description='Figure Name:', disabled=False)
output = widgets.Output()

display(wells, ymax1, ymax2, ymax3, figure_name, plot_button, output)

def plot_on_button_clicked(b):
    try:
        output.clear_output()

        os.makedirs("images", exist_ok=True)

        with output:
            Plot_OD_vs_time(wells.value, ymax1.value, figure_name.value)
            plt.savefig(f"images/{figure_name.value}_OD_vs_time.pdf", bbox_inches="tight")

            Plot_GFP_vs_time(wells.value, ymax2.value, figure_name.value)
            plt.savefig(f"images/{figure_name.value}_GFP_vs_time.pdf", bbox_inches="tight")

            Plot_GFP_vs_OD(wells.value, ymax3.value, figure_name.value)
            plt.savefig(f"images/{figure_name.value}_GFP_vs_OD.pdf", bbox_inches="tight")

            print("Plots saved in images/ folder")

    except:
        with output:
            print("Error!")
    return None

plot_button.on_click(plot_on_button_clicked)

SelectMultiple(description='Select Wells:', layout=Layout(height='600px'), options=('D2: pAH21', 'D3: pAH14', …

IntText(value=0, description='No.1 ymax:')

IntText(value=0, description='No.2 ymax:')

IntText(value=0, description='No.3 ymax:')

Text(value='', description='Figure Name:')

Button(description='Plot', style=ButtonStyle())

Output()

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

Linerae regression to compare plate reader to sort seq

In [ ]:
StatsWell('E4')

(np.float64(1.3524149633886762),
 np.float64(30.75153111984929),
 np.float64(0.9903758454845359),
 451)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

PATH = "flow_sort_seq.csv"

# -----------------------------
# 1. Load CSV
# -----------------------------
df = pd.read_csv(PATH)
df.columns = df.columns.str.strip().str.replace(r"\s+", "_", regex=True)

plate_col = "Expression_plate_reader"
sortseq_col = "Expression_sort_seq"
name_col = "Index"      # adjust if your ID column is called differently

# -----------------------------
# 2. Remove outlier BEFORE regression
# -----------------------------
# Option A: remove by ID (cleanest)
df = df[df[name_col] != "pAH01"]

# Option B: remove by numeric rule (if needed)
# df = df[df[sortseq_col] > 0.5]

# -----------------------------
# 3. Extract values for regression
# -----------------------------
X = df[plate_col].values.reshape(-1, 1)
y = df[sortseq_col].values

mask = ~np.isnan(X).flatten() & ~np.isnan(y)
X = X[mask]
y = y[mask]

# -----------------------------
# 4. Fit regression WITHOUT outlier
# -----------------------------
reg = LinearRegression().fit(X, y)

a = reg.coef_[0]
b = reg.intercept_

print("Slope a =", a)
print("Intercept b =", b)

# -----------------------------
# 5. Apply correction
# -----------------------------
df.loc[mask, "Plate_corrected"] = (a * df.loc[mask, plate_col] + b)

# -----------------------------
# 6. Save corrected results
# -----------------------------
df.to_csv("flow_sort_seq_corrected.csv", index=False)

print("\n✅ Done. Outlier removed and corrected values added.")

FileNotFoundError: [Errno 2] No such file or directory: 'flow_sort_seq.csv'

df["plate_corrected"] = a * df[Expression_plate_reader] + b

In [ ]:
df["plate_corrected"] = a * df[Expression_plate_reader] + b

NameError: name 'Expression_plate_reader' is not defined

evaluate plot